# Whisper Ultra-Fast: Transcripción de Alto Rendimiento

Este cuaderno utiliza **Faster-Whisper** con el modelo **large-v3**. Es hasta 5 veces más rápido que la versión estándar y está optimizado para audios de larga duración (3+ horas) en Kaggle.

In [ ]:
# 1. Instalación de Faster-Whisper
!pip install faster-whisper
!apt-get install -y ffmpeg

In [ ]:
from faster_whisper import WhisperModel
import torch
import os
import datetime
from pathlib import Path

# --- CONFIGURACIÓN ---
INPUT_DIR = "/kaggle/input/datasets/danieldobles/test-whisper"
OUTPUT_DIR = "/kaggle/working/transcriptions"
MODEL_NAME = "large-v3"
AUDIO_EXTENSIONS = (".mp3", ".wav", ".m4a", ".flac", ".ogg", ".mp4")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# VERIFICACIÓN CRÍTICA DE GPU
if not torch.cuda.is_available():
    raise RuntimeError("¡ERROR: GPU no detectada! Ve a 'Settings' -> 'Accelerator' y activa 'GPU T4 x2' o 'P100'.")

device = "cuda"
print(f"[OK] GPU Detectada. Usando {torch.cuda.get_device_name(0)}")

# --- CARGA DEL MODELO OPTIMIZADO ---
print(f"Cargando modelo {MODEL_NAME} en modo Faster...")
# compute_type="float16" es la clave para la velocidad en GPU
model = WhisperModel(MODEL_NAME, device=device, compute_type="float16")
print("Modelo cargado correctamente.")

In [ ]:
def transcribe_file_fast(file_path):
    """Transcribe usando faster-whisper con generador de segmentos."""
    file_path = Path(file_path)
    output_base = Path(OUTPUT_DIR) / file_path.stem
    
    if os.path.exists(f"{output_base}.txt"):
        print(f"[OMITIDO] Ya existe: {file_path.name}")
        return

    print(f"\n[PROCESANDO] {file_path}")
    start_time = datetime.datetime.now()
    
    try:
        # Transcribir (Faster-Whisper devuelve un generador)
        segments, info = model.transcribe(
            str(file_path), 
            beam_size=5, 
            language="es", # Forzamos español para ganar velocidad
            vad_filter=True, # Filtro de voz inteligente (ignora silencios largos)
            vad_parameters=dict(min_silence_duration_ms=500)
        )
        
        print(f"Idioma detectado: {info.language} con probabilidad {info.language_probability:.2f}")
        
        full_text = []
        for segment in segments:
            # Imprimir progreso con marca de tiempo
            ts = str(datetime.timedelta(seconds=int(segment.start)))
            print(f"[{ts}] {segment.text}")
            full_text.append(segment.text)
        
        final_text = " ".join(full_text)
        
        # Guardar resultados
        with open(f"{output_base}.txt", "w", encoding="utf-8") as f:
            f.write(final_text)
        with open(f"{output_base}.md", "w", encoding="utf-8") as f:
            f.write(f"# Transcripción: {file_path.name}\n\n{final_text}")
            
        duration = datetime.datetime.now() - start_time
        print(f"[OK] {file_path.name} terminado en {duration}")
        
    except Exception as e:
        print(f"[ERROR] Falló {file_path.name}: {e}")

# --- BUCLE PRINCIPAL ---
files_to_process = []
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(AUDIO_EXTENSIONS):
            files_to_process.append(os.path.join(root, file))

print(f"Archivos encontrados: {len(files_to_process)}")
for audio_file in files_to_process:
    transcribe_file_fast(audio_file)

print("\n--- PROCESO FINALIZADO ---")